# TradeFlow AI — Step 5: Agentic AI Orchestration

This notebook demonstrates the complete agentic AI workflow that coordinates:

1. **Forecasting Agent** — retrieves XGBoost demand forecasts
2. **Inventory Risk Agent** — analyzes shortages, excess stock, ageing risks
3. **Optimization Agent** — runs MILP transfer optimization (scipy.optimize.milp)
4. **Explanation Agent** — generates deterministic, business-friendly explanations
5. **Approval Agent** — manages human approval workflow with SQLite persistence
6. **Orchestrator** — coordinates all agents, supports individual agent calls

### Architecture

```
tradeflow_agents/
├── __init__.py
├── database.py            # SQLite persistent storage
├── forecast_agent.py      # Demand forecast retrieval
├── inventory_agent.py     # Inventory risk analysis
├── optimization_agent.py  # MILP transfer optimization
├── explanation_agent.py   # Deterministic explanation generation
├── approval_agent.py     # Human approval workflow
└── orchestrator.py        # Agent coordinator
```

### Key Design Principles

- Agents perform **actual operations** — no hardcoded responses
- All economic benefits are **projected estimates** from simulation
- Approvals are **never automatic** — every transfer starts as `Pending`
- Decisions **persist across restarts** via SQLite
- Scenario results are **kept separate** (original vs demonstration)


In [1]:
import sys
import os
import warnings
warnings.filterwarnings("ignore")

# Add workspace to path
sys.path.insert(0, r"c:\Users\antho\CodeBuddy\20261027105634")

DATA_DIR = r"C:\Users\antho\Desktop\CUD\Semester 7\Hackathon\Problem statement\Datasets"

# Import the agent package
from tradeflow_agents import (
    AgentDatabase,
    ForecastingAgent,
    InventoryRiskAgent,
    OptimizationAgent,
    ExplanationAgent,
    ApprovalAgent,
    OrchestratorAgent,
)

import pandas as pd
import numpy as np

print("TradeFlow AI — Step 5: Agentic AI Orchestration")
print("=" * 55)
print(f"Data directory: {DATA_DIR}")
print(f"Agent package: tradeflow_agents (v1.0.0)")
print()

# Verify all required data files exist
required_files = [
    "improved_daily_forecasts.csv",
    "improved_forecast_30d.csv",
    "scenario_inventory.csv",
    "scenario_inventory_risks.csv",
    "scenario_daily_inventory_projection.csv",
    "scenario_inbound_orders.csv",
    "transfer_routes.csv",
    "products.csv",
    "warehouses.csv",
    "transfer_recommendations.csv",
    "post_transfer_inventory_projection.csv",
    "transfer_optimization_summary.csv",
]

for f in required_files:
    path = os.path.join(DATA_DIR, f)
    status = "OK" if os.path.exists(path) else "MISSING"
    print(f"  [{status}] {f}")

print()
print("All modules imported successfully.")


TradeFlow AI — Step 5: Agentic AI Orchestration
Data directory: C:\Users\antho\Desktop\CUD\Semester 7\Hackathon\Problem statement\Datasets
Agent package: tradeflow_agents (v1.0.0)

  [OK] improved_daily_forecasts.csv
  [OK] improved_forecast_30d.csv
  [OK] scenario_inventory.csv
  [OK] scenario_inventory_risks.csv
  [OK] scenario_daily_inventory_projection.csv
  [OK] scenario_inbound_orders.csv
  [OK] transfer_routes.csv
  [OK] products.csv
  [OK] warehouses.csv
  [OK] transfer_recommendations.csv
  [OK] post_transfer_inventory_projection.csv
  [OK] transfer_optimization_summary.csv

All modules imported successfully.


## 1. Initialize the Agent System

The Orchestrator initializes all specialized agents and the SQLite database.
The database persists across restarts — approval decisions are never lost.


In [2]:
# Use a persistent DB in the Datasets directory
DB_PATH = os.path.join(DATA_DIR, "tradeflow_agents.db")

# Clean start for demo (remove old DB)
if os.path.exists(DB_PATH):
    os.remove(DB_PATH)

# Initialize the orchestrator — this creates all agents
orchestrator = OrchestratorAgent(DATA_DIR, db_path=DB_PATH)

print("Agent System Initialized")
print("=" * 55)
print(f"  Orchestrator:      {orchestrator.name}")
print(f"  ForecastingAgent:  {orchestrator.forecast_agent.name}")
print(f"  InventoryRiskAgent:{orchestrator.inventory_agent.name}")
print(f"  OptimizationAgent: {orchestrator.optimization_agent.name}")
print(f"  ExplanationAgent:  {orchestrator.explanation_agent.name}")
print(f"  ApprovalAgent:     {orchestrator.approval_agent.name}")
print(f"  Database:          {orchestrator.db.db_path}")
print()
print("All agents are ready and can be called individually.")


Agent System Initialized
  Orchestrator:      OrchestratorAgent
  ForecastingAgent:  ForecastingAgent
  InventoryRiskAgent:InventoryRiskAgent
  OptimizationAgent: OptimizationAgent
  ExplanationAgent:  ExplanationAgent
  ApprovalAgent:     ApprovalAgent
  Database:          C:\Users\antho\Desktop\CUD\Semester 7\Hackathon\Problem statement\Datasets\tradeflow_agents.db

All agents are ready and can be called individually.


## 2. Forecasting Agent — Retrieve Demand Predictions

The Forecasting Agent loads the improved XGBoost forecasts and provides:
- Per-SKU/warehouse demand forecasts (30-day aggregate + daily)
- Forecast uncertainty (lower/upper bounds)
- Summary statistics across all combinations
- Model limitation explanations


In [3]:
# --- Retrieve forecast for a specific SKU-warehouse combination ---
sku = "SKU-001"
warehouse = "Dubai"

forecast = orchestrator.get_demand_forecast(sku, warehouse)

print(f"Forecast for {sku} at {warehouse}")
print("-" * 50)
print(f"  30-Day Forecast:   {forecast['forecast_30d']:.2f} units")
print(f"  Lower Bound:       {forecast['lower_bound']:.2f} units")
print(f"  Upper Bound:       {forecast['upper_bound']:.2f} units")
print(f"  Selected Model:   {forecast['selected_model']}")
print(f"  Avg Daily Demand:  {forecast['avg_daily_demand']:.2f} units/day")
print(f"  Max Daily Demand:  {forecast['max_daily_demand']:.2f} units/day")
print(f"  Min Daily Demand:  {forecast['min_daily_demand']:.2f} units/day")

# Show first 10 days of daily forecast
print()
print("  First 10 days of daily forecast:")
for d in forecast['daily_forecast'][:10]:
    print(f"    {d['date']}: {d['predicted_demand']:.2f}")


Forecast for SKU-001 at Dubai
--------------------------------------------------
  30-Day Forecast:   121.52 units
  Lower Bound:       114.72 units
  Upper Bound:       128.32 units
  Selected Model:   Tuned XGBoost
  Avg Daily Demand:  4.05 units/day
  Max Daily Demand:  4.52 units/day
  Min Daily Demand:  3.34 units/day

  First 10 days of daily forecast:
    2026-09-29: 4.52
    2026-09-30: 4.44
    2026-10-01: 4.44
    2026-10-02: 4.46
    2026-10-03: 3.58
    2026-10-04: 3.34
    2026-10-05: 4.48
    2026-10-06: 4.18
    2026-10-07: 4.18
    2026-10-08: 4.26


In [4]:
# --- Forecast summary across all SKU-warehouse combinations ---
summary = orchestrator.forecast_agent.summarize_forecast()

print("Forecast Summary (All Combinations)")
print("-" * 50)
print(f"  Total 30-day demand:  {summary['total_30d_demand']:.0f} units")
print(f"  Combinations:          {summary['num_combos']}")
print(f"  Avg per combo:         {summary['avg_30d_per_combo']:.1f} units")
print(f"  Max 30-day:            {summary['max_30d']:.0f} units")
print(f"  Min 30-day:            {summary['min_30d']:.0f} units")
print(f"  Avg uncertainty range: {summary['avg_uncertainty_range']:.2f} units")
print(f"  Avg uncertainty pct:   {summary['avg_uncertainty_pct']:.1%}")

print()
print("  By Warehouse:")
for wh, data in summary['by_warehouse'].items():
    print(f"    {wh}: total={data['total']:.0f}, avg={data['avg']:.1f}, combos={data['count']}")


Forecast Summary (All Combinations)
--------------------------------------------------
  Total 30-day demand:  6385 units
  Combinations:          60
  Avg per combo:         106.4 units
  Max 30-day:            160 units
  Min 30-day:            62 units
  Avg uncertainty range: 13.60 units
  Avg uncertainty pct:   13.9%

  By Warehouse:
    Abu Dhabi: total=2189, avg=109.4, combos=20
    Dubai: total=2456, avg=122.8, combos=20
    Sharjah: total=1741, avg=87.0, combos=20


In [5]:
# --- Model limitations ---
limitations = orchestrator.forecast_agent.explain_model_limitations()

print("Model Limitations")
print("-" * 50)
print(f"  Model: {limitations['model_type']}")
print()
print("  Key limitations:")
for i, lim in enumerate(limitations['key_limitations'], 1):
    print(f"    {i}. {lim}")
print()
print(f"  Note: {limitations['uncertainty_note']}")
print(f"  Recommendation: {limitations['recommendation']}")


Model Limitations
--------------------------------------------------
  Model: XGBoost (improved with hyperparameter tuning)

  Key limitations:
    1. Forecasts are based on historical sales patterns and may not capture sudden market shifts.
    2. Prediction intervals (lower/upper bounds) reflect statistical uncertainty, not operational risk.
    3. Forecast accuracy degrades for SKUs with sparse or highly variable sales history.
    4. The model does not account for promotional events, supply disruptions, or competitor actions.
    5. 30-day aggregate forecasts accumulate daily errors — confidence narrows for longer horizons.

  Note: The lower and upper bounds represent prediction intervals. Actual demand may fall outside this range in rare cases.
  Recommendation: Use forecasts as planning guidance, not certainties. Monitor actual demand vs forecast and retrain periodically.


## 3. Inventory Risk Agent — Analyze Inventory Against Forecasts

The Inventory Risk Agent:
- Identifies critical shortages (available < forecast demand)
- Identifies excess stock and ageing risks
- Compares original vs demonstration scenarios
- Returns structured risk reports with recommendations


In [6]:
# --- Analyze demonstration scenario ---
demo_analysis = orchestrator.analyze_inventory("demonstration")

print("Demonstration Scenario — Inventory Risk Analysis")
print("=" * 55)
print(f"  Total combinations:     {demo_analysis['total_combos']}")
print(f"  Total available stock:  {demo_analysis['total_available_stock']:,} units")
print(f"  Total 30-day forecast:  {demo_analysis['total_forecast_30d']:.0f} units")
print(f"  Coverage ratio:         {demo_analysis['coverage_ratio']:.2f}x")
print(f"  Total unmet demand:     {demo_analysis['total_unmet_demand']:.0f} units")
print(f"  Total excess stock:     {demo_analysis['total_excess_stock']:.0f} units")
print()
print("  Risk Distribution:")
for risk, count in demo_analysis['risk_counts'].items():
    print(f"    {risk}: {count}")
print()
print("  By Warehouse:")
for wh, data in demo_analysis['by_warehouse'].items():
    print(f"    {wh}: critical={data['critical']}, excess={data['excess']}, "
          f"unmet={data['total_unmet']:.0f}, excess_qty={data['total_excess']:.0f}")


Demonstration Scenario — Inventory Risk Analysis
  Total combinations:     60
  Total available stock:  10,936 units
  Total 30-day forecast:  6385 units
  Coverage ratio:         1.71x
  Total unmet demand:     1351 units
  Total excess stock:     5385 units

  Risk Distribution:
    Critical: 32
    Excess: 19
    Healthy: 9

  By Warehouse:
    Abu Dhabi: critical=10, excess=6, unmet=398, excess_qty=2143
    Dubai: critical=10, excess=7, unmet=576, excess_qty=1861
    Sharjah: critical=12, excess=6, unmet=377, excess_qty=1382


In [7]:
# --- Analyze original scenario ---
orig_analysis = orchestrator.analyze_inventory("original")

print("Original Scenario — Inventory Risk Analysis")
print("=" * 55)
print(f"  Total combinations:     {orig_analysis['total_combos']}")
print(f"  Total available stock:  {orig_analysis['total_available_stock']:,} units")
print(f"  Total 30-day forecast:  {orig_analysis['total_forecast_30d']:.0f} units")
print(f"  Coverage ratio:         {orig_analysis['coverage_ratio']:.2f}x")
print(f"  Total unmet demand:     {orig_analysis['total_unmet_demand']:.0f} units")
print(f"  Total excess stock:     {orig_analysis['total_excess_stock']:.0f} units")
print()
print("  Risk Distribution:")
for risk, count in orig_analysis['risk_counts'].items():
    print(f"    {risk}: {count}")


Original Scenario — Inventory Risk Analysis
  Total combinations:     60
  Total available stock:  40,872 units
  Total 30-day forecast:  6385 units
  Coverage ratio:         6.40x
  Total unmet demand:     0 units
  Total excess stock:     32937 units

  Risk Distribution:
    Excess: 58
    Healthy: 2


In [8]:
# --- Compare scenarios ---
comparison = orchestrator.compare_scenarios()

print("Scenario Comparison")
print("=" * 55)
print(f"  {'Metric':<30} {'Original':>15} {'Demonstration':>15}")
print(f"  {'-'*60}")
print(f"  {'Total unmet demand':<30} {comparison['original']['total_unmet']:>15.0f} {comparison['demonstration']['total_unmet']:>15.0f}")
print(f"  {'Total excess stock':<30} {comparison['original']['total_excess']:>15.0f} {comparison['demonstration']['total_excess']:>15.0f}")
print(f"  {'Coverage ratio':<30} {comparison['original']['coverage_ratio']:>15.2f}x {comparison['demonstration']['coverage_ratio']:>15.2f}x")
print()
print(f"  Note: {comparison['comparison']['note']}")


Scenario Comparison
  Metric                                Original   Demonstration
  ------------------------------------------------------------
  Total unmet demand                           0            1351
  Total excess stock                       32937            5385
  Coverage ratio                            6.40x            1.71x

  Note: The demonstration scenario was designed to show shortages and transfer opportunities. The original scenario has heavy overstocking and ageing.


## 4. Optimization Agent — Run Transfer Optimization

The Optimization Agent:
- Loads the Step 4 optimization logic (refactored into reusable functions)
- Runs scipy.optimize.milp with integer decision variables
- Validates source inventory availability
- Respects route lead times (transfers arriving after stockout are excluded)
- Avoids double allocation of source inventory
- Simulates pre/post-transfer inventory daily


In [9]:
# --- Run optimization for demonstration scenario ---
opt_result = orchestrator.get_transfer_recommendations("demonstration")

print("Optimization Results — Demonstration Scenario")
print("=" * 55)
print(f"  Recommendations generated:  {opt_result['num_recommendations']}")
print(f"  Total units recommended:   {opt_result['total_units_recommended']:,}")
print(f"  Total transfer cost:        {opt_result['total_transfer_cost']:,.2f} AED")
print()
print(f"  Baseline unmet demand:      {opt_result['baseline_unmet']:.1f} units")
print(f"  Post-transfer unmet:        {opt_result['post_transfer_unmet']:.1f} units")
print(f"  Unmet demand avoided:       {opt_result['unmet_avoided']:.1f} units")
print(f"  Reduction percentage:       {opt_result['unmet_reduction_pct']:.1f}%")
print()
print(f"  Actual margin preserved:    {opt_result['actual_margin_preserved']:,.2f} AED")
print(f"  Net economic benefit:      {opt_result['net_economic_benefit']:,.2f} AED")
print()
print(f"  Shortages before:           {opt_result['shortages_before']}")
print(f"  Shortages after:            {opt_result['shortages_after']}")
print(f"  Fully eliminated:           {opt_result['shortages_fully_eliminated']}")


Optimization Results — Demonstration Scenario
  Recommendations generated:  28
  Total units recommended:   1,108
  Total transfer cost:        2,021.50 AED

  Baseline unmet demand:      1119.7 units
  Post-transfer unmet:        12.4 units
  Unmet demand avoided:       1107.3 units
  Reduction percentage:       98.9%

  Actual margin preserved:    46,838.67 AED
  Net economic benefit:      44,817.17 AED

  Shortages before:           26
  Shortages after:            24
  Fully eliminated:           2


In [10]:
# --- Show top 10 recommendations by net benefit ---
recs_df = pd.DataFrame(opt_result['recommendations'])
print("Top 10 Transfer Recommendations (by Net Benefit)")
print("=" * 100)
print(recs_df[['recommendation_id', 'sku', 'source_warehouse',
               'destination_warehouse', 'transfer_quantity', 'lead_time_days',
               'transfer_cost_aed', 'expected_unmet_avoided',
               'estimated_net_benefit_aed']].head(10).to_string(index=False))


Top 10 Transfer Recommendations (by Net Benefit)
      recommendation_id     sku source_warehouse destination_warehouse  transfer_quantity  lead_time_days  transfer_cost_aed  expected_unmet_avoided  estimated_net_benefit_aed
TRF-DEM-SKU-016-SHA-ABU SKU-016          Sharjah             Abu Dhabi                 69               2              172.5                      69                    4905.90
TRF-DEM-SKU-010-SHA-DUB SKU-010          Sharjah                 Dubai                 72               1               72.0                      72                    4514.40
TRF-DEM-SKU-007-DUB-ABU SKU-007            Dubai             Abu Dhabi                 56               1              112.0                      56                    3193.12
TRF-DEM-SKU-002-ABU-SHA SKU-002        Abu Dhabi               Sharjah                 45               2              112.5                      45                    3003.30
TRF-DEM-SKU-011-DUB-SHA SKU-011            Dubai               Sharjah 

In [11]:
# --- Run optimization for original scenario (no shortages expected) ---
orig_opt = orchestrator.get_transfer_recommendations("original")

print("Optimization Results — Original Scenario")
print("=" * 55)
print(f"  Has recommendations:    {orig_opt['has_recommendations']}")
print(f"  Num recommendations:   {orig_opt['num_recommendations']}")
print(f"  Baseline unmet:        {orig_opt['baseline_unmet']:.1f}")
print(f"  Message:               {orig_opt.get('message', 'N/A')}")
print()
print("  The original scenario has heavy overstocking and no shortages,")
print("  so no transfers are recommended. This is expected behavior.")


Optimization Results — Original Scenario
  Has recommendations:    False
  Num recommendations:   0
  Baseline unmet:        0.0
  Message:               No shortages detected in original scenario. No transfers needed.

  The original scenario has heavy overstocking and no shortages,
  so no transfers are recommended. This is expected behavior.


In [12]:
# --- Validate source inventory (no double allocation) ---
validation = orchestrator.optimization_agent.validate_source_inventory(
    opt_result['recommendations']
)

print("Source Inventory Validation")
print("=" * 55)
print(f"  Valid:  {validation['valid']}")
print(f"  Issues: {len(validation['issues'])}")
if validation['issues']:
    for issue in validation['issues']:
        print(f"    - {issue}")
else:
    print("  All source warehouses have sufficient stock.")
print()
print("  Source usage:")
for source, qty in validation['source_usage'].items():
    print(f"    {source}: {qty} units allocated")


Source Inventory Validation
  Valid:  True
  Issues: 0
  All source warehouses have sufficient stock.

  Source usage:
    SKU-016/Sharjah: 71 units allocated
    SKU-010/Sharjah: 72 units allocated
    SKU-007/Dubai: 56 units allocated
    SKU-002/Abu Dhabi: 45 units allocated
    SKU-011/Dubai: 65 units allocated
    SKU-013/Sharjah: 67 units allocated
    SKU-020/Dubai: 97 units allocated
    SKU-006/Dubai: 50 units allocated
    SKU-009/Abu Dhabi: 115 units allocated
    SKU-003/Abu Dhabi: 116 units allocated
    SKU-012/Sharjah: 15 units allocated
    SKU-017/Dubai: 44 units allocated
    SKU-018/Abu Dhabi: 48 units allocated
    SKU-005/Dubai: 77 units allocated
    SKU-019/Abu Dhabi: 81 units allocated
    SKU-008/Abu Dhabi: 34 units allocated
    SKU-015/Dubai: 38 units allocated
    SKU-014/Sharjah: 15 units allocated
    SKU-010/Abu Dhabi: 1 units allocated
    SKU-007/Sharjah: 1 units allocated


## 5. Explanation Agent — Generate Business-Friendly Explanations

The Explanation Agent generates **deterministic** explanations using actual
optimization results. It does NOT invent financial figures. All numbers come
directly from the optimization output.

If an LLM API is unavailable, deterministic explanation generation is used as
the fallback. All explanations are clearly labeled as deterministic.


In [13]:
# --- Explain a specific transfer ---
first_rec = opt_result['recommendations'][0]
tid = first_rec['recommendation_id']

explanation = orchestrator.explain_transfer(tid)

print(f"Transfer Explanation: {tid}")
print("=" * 70)
print()
print(f"SKU:                    {explanation['sku']}")
print(f"Product:                {explanation['product_name']}")
print(f"Source:                 {explanation['source_warehouse']}")
print(f"Destination:            {explanation['destination_warehouse']}")
print(f"Quantity:               {explanation['transfer_quantity']} units")
print(f"Lead time:              {explanation['lead_time_days']} day(s)")
print(f"Transfer cost:          {explanation['transfer_cost_aed']:.2f} AED")
print(f"Cost per unit:          {explanation['cost_per_unit_aed']:.2f} AED")
print(f"Expected unmet avoided: {explanation['expected_unmet_avoided']:.0f} units")
print(f"Profit margin:          {explanation['profit_margin_aed']:.2f} AED/unit")
print(f"Margin preserved:       {explanation['projected_margin_preserved']:.2f} AED")
print(f"Net benefit:            {explanation['estimated_net_benefit_aed']:.2f} AED")
print(f"Explanation type:       {explanation['explanation_type']}")
print()
print("Source Selection Reason:")
print(f"  {explanation['source_selection_reason']}")
print()
print("Shortage Reduction Reason:")
print(f"  {explanation['shortage_reduction_reason']}")
print()
print("Cost-Benefit Reason:")
print(f"  {explanation['cost_benefit_reason']}")
print()
print("Assumptions:")
print(f"  {explanation['assumptions']}")
print()
print("Uncertainty:")
print(f"  {explanation['uncertainty']}")
print()
print("Recommendation Summary:")
print(f"  {explanation['reason_for_recommendation']}")


Transfer Explanation: TRF-DEM-SKU-016-SHA-ABU

SKU:                    SKU-016
Product:                Memory Card
Source:                 Sharjah
Destination:            Abu Dhabi
Quantity:               69 units
Lead time:              2 day(s)
Transfer cost:          172.50 AED
Cost per unit:          2.50 AED
Expected unmet avoided: 69 units
Profit margin:          73.60 AED/unit
Margin preserved:       5078.40 AED
Net benefit:            4905.90 AED
Explanation type:       deterministic

Source Selection Reason:
  Sharjah was selected as the source because it holds transferable surplus stock of Memory Card (SKU-016) — inventory above its own forecast demand and safety stock requirements. Transferring from Sharjah does not create new shortages there.

Shortage Reduction Reason:
  Abu Dhabi faces a projected shortage of 69 units over the 30-day planning horizon. Transferring 69 units from Sharjah is expected to reduce this unmet demand by approximately 69 units (capped at the reduci

In [14]:
# --- System summary explanation ---
sys_explanation = orchestrator.explanation_agent.explain_system_summary(opt_result)

print("System Summary Explanation")
print("=" * 70)
print(f"Type: {sys_explanation['summary_type']}")
print()
print(sys_explanation['summary'])
print()
print("Key Metrics:")
for k, v in sys_explanation['key_metrics'].items():
    if isinstance(v, float):
        print(f"  {k}: {v:.2f}")
    else:
        print(f"  {k}: {v}")
print()
print(f"Note: {sys_explanation['note']}")


System Summary Explanation
Type: deterministic

The optimization agent identified 28 transfer recommendations for the demonstration scenario. These transfers are projected to reduce unmet demand from 1120 to 12 units (98.9% reduction). 2 of 26 shortage combos are projected to be fully eliminated; the remainder are significantly reduced but still have small residual unmet demand due to transfer lead times. Total transfer cost: 2021.50 AED. Projected net economic benefit: 44817.17 AED. All benefits are projected estimates and require human approval.

Key Metrics:
  num_recommendations: 28
  baseline_unmet: 1119.69
  post_transfer_unmet: 12.41
  unmet_avoided: 1107.28
  reduction_pct: 98.89
  total_transfer_cost: 2021.50
  net_economic_benefit: 44817.17
  shortages_fully_eliminated: 2

Note: This is a deterministic summary generated from actual optimization results.


## 6. Approval Agent — Human Approval Workflow

The Approval Agent:
- Presents proposed transfers for human approval
- All recommendations start as `Pending`
- Supports approve and reject operations
- Prevents duplicate approvals/rejections
- Records decisions with timestamps
- Maintains an audit trail in SQLite
- Decisions persist across restarts
- Does NOT execute physical warehouse transfers


In [15]:
# --- Get pending approvals ---
pending = orchestrator.approval_agent.get_pending_approvals(scenario="demonstration")

print(f"Pending Approvals: {len(pending)}")
print("=" * 55)
print(f"{'ID':<35} {'SKU':<10} {'Qty':>5} {'Benefit':>12}")
print("-" * 65)
for p in pending[:10]:
    print(f"{p['recommendation_id']:<35} {p['sku']:<10} {p['transfer_quantity']:>5} {p['estimated_net_benefit_aed']:>12.2f}")
if len(pending) > 10:
    print(f"... and {len(pending) - 10} more")
print()
print("All recommendations start as 'Pending'. No transfer is auto-approved.")


Pending Approvals: 28
ID                                  SKU          Qty      Benefit
-----------------------------------------------------------------
TRF-DEM-SKU-016-SHA-ABU             SKU-016       69      4905.90
TRF-DEM-SKU-010-SHA-DUB             SKU-010       72      4514.40
TRF-DEM-SKU-007-DUB-ABU             SKU-007       56      3193.12
TRF-DEM-SKU-002-ABU-SHA             SKU-002       45      3003.30
TRF-DEM-SKU-011-DUB-SHA             SKU-011       64      2878.72
TRF-DEM-SKU-013-SHA-DUB             SKU-013       57      2187.09
TRF-DEM-SKU-020-DUB-ABU             SKU-020       97      2098.11
TRF-DEM-SKU-006-DUB-SHA             SKU-006       50      1969.00
TRF-DEM-SKU-009-ABU-DUB             SKU-009       82      1866.32
TRF-DEM-SKU-003-ABU-DUB             SKU-003      116      1825.84
... and 18 more

All recommendations start as 'Pending'. No transfer is auto-approved.


In [16]:
# --- Demonstrate approval workflow ---
# Approve the top recommendation
top_tid = opt_result['recommendations'][0]['recommendation_id']
approve_result = orchestrator.approve_transfer(top_tid, decided_by="demo_user",
                                                note="High net benefit transfer")

print("Approval Action")
print("=" * 55)
print(f"  Transfer ID:    {top_tid}")
print(f"  Success:        {approve_result['success']}")
print(f"  Previous status: {approve_result.get('previous_status', 'N/A')}")
print(f"  New status:     {approve_result.get('new_status', 'N/A')}")
print(f"  Decided by:     {approve_result.get('decided_by', 'N/A')}")
print(f"  Timestamp:      {approve_result.get('timestamp', 'N/A')}")
print(f"  Message:        {approve_result.get('message', 'N/A')}")

print()

# Try to approve the same transfer again (should be prevented)
duplicate_result = orchestrator.approve_transfer(top_tid)
print("Duplicate Approval Attempt")
print("=" * 55)
print(f"  Success:  {duplicate_result['success']}")
print(f"  Error:    {duplicate_result.get('error', 'N/A')}")
print(f"  Message:  {duplicate_result.get('message', 'N/A')}")
print("  (Duplicate approval correctly prevented)")

print()

# Try to reject the already-approved transfer (should be prevented)
reject_result = orchestrator.reject_transfer(top_tid)
print("Reject After Approve Attempt")
print("=" * 55)
print(f"  Success:  {reject_result['success']}")
print(f"  Error:    {reject_result.get('error', 'N/A')}")
print(f"  Message:  {reject_result.get('message', 'N/A')}")
print("  (Cannot override approval — correctly prevented)")


Approval Action
  Transfer ID:    TRF-DEM-SKU-016-SHA-ABU
  Success:        True
  Previous status: Pending
  New status:     Approved
  Decided by:     demo_user
  Timestamp:      2026-10-27T15:14:47.022013
  Message:        Transfer TRF-DEM-SKU-016-SHA-ABU approved successfully.

Duplicate Approval Attempt
  Success:  False
  Error:    Duplicate approval prevented
  Message:  Transfer TRF-DEM-SKU-016-SHA-ABU is already approved.
  (Duplicate approval correctly prevented)

Reject After Approve Attempt
  Success:  False
  Error:    Cannot reject an approved transfer
  Message:  Transfer TRF-DEM-SKU-016-SHA-ABU was already approved. Approval decisions are final.
  (Cannot override approval — correctly prevented)


In [17]:
# --- Reject a different transfer ---
if len(opt_result['recommendations']) > 1:
    second_tid = opt_result['recommendations'][1]['recommendation_id']
    reject_result = orchestrator.reject_transfer(second_tid, decided_by="demo_user",
                                                  note="Low priority")
    print("Rejection Action")
    print("=" * 55)
    print(f"  Transfer ID:    {second_tid}")
    print(f"  Success:        {reject_result['success']}")
    print(f"  New status:     {reject_result.get('new_status', 'N/A')}")
    print(f"  Message:        {reject_result.get('message', 'N/A')}")


Rejection Action
  Transfer ID:    TRF-DEM-SKU-010-SHA-DUB
  Success:        True
  New status:     Rejected
  Message:        Transfer TRF-DEM-SKU-010-SHA-DUB rejected successfully.


In [18]:
# --- Approval summary ---
approval_summary = orchestrator.approval_agent.get_approval_summary(scenario="demonstration")

print("Approval Summary — Demonstration Scenario")
print("=" * 55)
print(f"  Total recommendations: {approval_summary['total']}")
print(f"  Pending:              {approval_summary['pending']}")
print(f"  Approved:             {approval_summary['approved']}")
print(f"  Rejected:             {approval_summary['rejected']}")
print(f"  Approved units:       {approval_summary['approved_units']}")
print(f"  Approved cost:        {approval_summary['approved_transfer_cost']:.2f} AED")
print(f"  Approved benefit:     {approval_summary['approved_net_benefit']:.2f} AED")


Approval Summary — Demonstration Scenario
  Total recommendations: 28
  Pending:              26
  Approved:             1
  Rejected:             1
  Approved units:       69
  Approved cost:        172.50 AED
  Approved benefit:     4905.90 AED


In [19]:
# --- Audit trail ---
audit = orchestrator.get_audit_trail(limit=15)

print("Audit Trail (Last 15 entries)")
print("=" * 55)
print(f"{'Timestamp':<30} {'Agent':<20} {'Action':<25}")
print("-" * 75)
for a in audit[:15]:
    ts = a['timestamp'][:19]
    print(f"{ts:<30} {a['agent_name']:<20} {a['action']:<25}")


Audit Trail (Last 15 entries)
Timestamp                      Agent                Action                   
---------------------------------------------------------------------------
2026-10-27T15:14:47            ApprovalAgent        update_approval          
2026-10-27T15:14:47            ApprovalAgent        update_approval          
2026-10-27T15:14:46            ApprovalAgent        save_recommendation      
2026-10-27T15:14:46            ApprovalAgent        save_recommendation      
2026-10-27T15:14:46            ApprovalAgent        save_recommendation      
2026-10-27T15:14:46            ApprovalAgent        save_recommendation      
2026-10-27T15:14:46            ApprovalAgent        save_recommendation      
2026-10-27T15:14:46            ApprovalAgent        save_recommendation      
2026-10-27T15:14:46            ApprovalAgent        save_recommendation      
2026-10-27T15:14:46            ApprovalAgent        save_recommendation      
2026-10-27T15:14:46            Appro

## 7. Persistence — Decisions Survive Restarts

The SQLite database ensures approval decisions persist across sessions.
Let's verify by creating a new Orchestrator instance (simulating a restart).


In [20]:
# --- Simulate restart: create a new orchestrator ---
orchestrator2 = OrchestratorAgent(DATA_DIR, db_path=DB_PATH)

# Check that decisions persisted
status1 = orchestrator2.approval_agent.get_approval_status(top_tid)
print("Persistence Test")
print("=" * 55)
print(f"  Transfer:     {top_tid}")
print(f"  Status:       {status1['approval_status']}")
print(f"  Decided by:   {status1.get('decided_by', 'N/A')}")
print(f"  Timestamp:    {status1.get('decision_timestamp', 'N/A')[:19]}")
print()
print("  Decision persisted across restart!")

if len(opt_result['recommendations']) > 1:
    status2 = orchestrator2.approval_agent.get_approval_status(second_tid)
    print()
    print(f"  Transfer:     {second_tid}")
    print(f"  Status:       {status2['approval_status']}")
    print(f"  Decided by:   {status2.get('decided_by', 'N/A')}")
    print(f"  Timestamp:    {status2.get('decision_timestamp', 'N/A')[:19]}")
    print()
    print("  Rejection also persisted across restart!")


Persistence Test
  Transfer:     TRF-DEM-SKU-016-SHA-ABU
  Status:       Approved
  Decided by:   demo_user
  Timestamp:    2026-10-27T15:14:47

  Decision persisted across restart!

  Transfer:     TRF-DEM-SKU-010-SHA-DUB
  Status:       Rejected
  Decided by:   demo_user
  Timestamp:    2026-10-27T15:14:47

  Rejection also persisted across restart!


## 8. Scenario Isolation — Results Are Kept Separate

The system supports both scenarios independently. Switching scenarios does not mix results.


In [21]:
# --- Verify scenario isolation ---
demo_approvals = orchestrator2.approval_agent.get_all_approvals(scenario="demonstration")
orig_approvals = orchestrator2.approval_agent.get_all_approvals(scenario="original")

print("Scenario Isolation")
print("=" * 55)
print(f"  Demonstration approvals: {len(demo_approvals)}")
print(f"  Original approvals:      {len(orig_approvals)}")
print()

# Verify no ID overlap
demo_ids = {r['recommendation_id'] for r in demo_approvals}
orig_ids = {r['recommendation_id'] for r in orig_approvals}
overlap = demo_ids & orig_ids
print(f"  ID overlap: {len(overlap)} (should be 0)")
print()

# Verify demo IDs have DEM prefix
demo_prefix_ok = all("DEM" in rid for rid in demo_ids)
print(f"  Demo IDs have DEM prefix: {demo_prefix_ok}")
print()
print("  Scenarios are properly isolated.")


Scenario Isolation
  Demonstration approvals: 28
  Original approvals:      0

  ID overlap: 0 (should be 0)

  Demo IDs have DEM prefix: True

  Scenarios are properly isolated.


## 9. Full Workflow Execution

The Orchestrator can run the complete workflow end-to-end, or individual
agents can be called as needed.


In [22]:
# --- Run full workflow for demonstration scenario ---
# Using a fresh DB to avoid interfering with previous approval decisions
WF_DB = os.path.join(DATA_DIR, "tradeflow_workflow_demo.db")
if os.path.exists(WF_DB):
    os.remove(WF_DB)

orch_wf = OrchestratorAgent(DATA_DIR, db_path=WF_DB)

print("Running Full Workflow (Demonstration Scenario)")
print("=" * 55)
results = orch_wf.run_full_workflow("demonstration")

print(f"  Started:  {results['started_at'][:19]}")
print(f"  Completed:{results['completed_at'][:19]}")
print()
print(f"  Forecast summary:  {results['forecast']['total_30d_demand']:.0f} total demand")
print(f"  Inventory:         {results['inventory']['risk_counts']}")
print(f"  Optimization:      {results['optimization']['num_recommendations']} recommendations")
print(f"  Explanations:      {len(results.get('explanations', []))} generated")
print(f"  Approval loaded:   {results['approval_loaded']['loaded']} transfers")
print()
summary = results['summary']
print("  System Summary:")
print(f"    Baseline unmet:      {summary['optimization']['baseline_unmet']:.1f}")
print(f"    Post-transfer unmet: {summary['optimization']['post_transfer_unmet']:.1f}")
print(f"    Unmet avoided:       {summary['optimization']['unmet_avoided']:.1f}")
print(f"    Reduction:           {summary['optimization']['unmet_reduction_pct']:.1f}%")
print(f"    Net benefit:         {summary['optimization']['net_economic_benefit']:.2f} AED")
print(f"    Shortages eliminated:{summary['optimization']['shortages_fully_eliminated']}")
print()
print(f"  Explanation: {summary['explanation'][:200]}...")


Running Full Workflow (Demonstration Scenario)

  Started:  2026-10-27T15:14:47
  Completed:2026-10-27T15:14:47

  Forecast summary:  6385 total demand
  Inventory:         {'Critical': 32, 'Excess': 19, 'Healthy': 9}
  Optimization:      28 recommendations
  Explanations:      28 generated
  Approval loaded:   28 transfers

  System Summary:
    Baseline unmet:      1119.7
    Post-transfer unmet: 12.4
    Unmet avoided:       1107.3
    Reduction:           98.9%
    Net benefit:         44817.17 AED
    Shortages eliminated:2

  Explanation: The optimization agent identified 28 transfer recommendations for the demonstration scenario. These transfers are projected to reduce unmet demand from 1120 to 12 units (98.9% reduction). 2 of 26 shor...


## 10. Step 4 Discrepancy Documentation

Step 4 reported a 98.9% unmet-demand reduction but only 2 of 26 shortage combos
were fully eliminated. This is **expected behavior**, not a bug:

- **98.9% reduction** = aggregate units avoided (1,107.3 / 1,119.7)
- **2/26 fully eliminated** = combos with zero remaining unmet demand
- The remaining 24 combos were significantly reduced but still have small
  residual unmet demand because transfers arrive after lead time, so
  early-day shortages cannot be prevented


In [23]:
# --- Step 4 discrepancy documentation ---
print("Step 4 Discrepancy Explanation")
print("=" * 70)
print()
print(f"  Unmet demand reduction: {opt_result['unmet_reduction_pct']:.1f}%")
print(f"  Aggregate units avoided: {opt_result['unmet_avoided']:.1f} / {opt_result['baseline_unmet']:.1f}")
print(f"  Shortages before:       {opt_result['shortages_before']}")
print(f"  Shortages after:        {opt_result['shortages_after']}")
print(f"  Fully eliminated:       {opt_result['shortages_fully_eliminated']}")
print()
print("  Discrepancy Note:")
print(f"  {opt_result['step4_discrepancy_note']}")
print()
print("  Conclusion: The high aggregate reduction percentage reflects units saved,")
print("  not the count of fully eliminated shortage cases. Both metrics are correct")
print("  — they measure different things. This is expected behavior.")


Step 4 Discrepancy Explanation

  Unmet demand reduction: 98.9%
  Aggregate units avoided: 1107.3 / 1119.7
  Shortages before:       26
  Shortages after:        24
  Fully eliminated:       2

  Discrepancy Note:
  The 98.9% unmet-demand reduction reflects aggregate units avoided (1107.3/1119.7). Only 2 of 26 shortage combos were fully eliminated to zero. The remainder were significantly reduced but still have small residual unmet demand because transfers arrive after lead time, so early-day shortages cannot be prevented. This is expected — not a bug.

  Conclusion: The high aggregate reduction percentage reflects units saved,
  not the count of fully eliminated shortage cases. Both metrics are correct
  — they measure different things. This is expected behavior.


## 11. Error Handling — Graceful Failure

The agent system handles errors gracefully:
- Invalid scenarios return structured error messages
- Invalid SKU lookups return `found: False`
- Invalid transfer IDs are rejected
- Agent errors are logged to the database


In [24]:
# --- Test error handling ---
print("Error Handling Tests")
print("=" * 55)

# Invalid scenario
result = orchestrator2.analyze_inventory("invalid_scenario")
print(f"  Invalid scenario: {result.get('error', 'N/A')}")

# Invalid SKU
result = orchestrator2.get_demand_forecast("SKU-NONEXIST", "Dubai")
print(f"  Invalid SKU: {result.get('error', 'N/A')}")

# Invalid transfer ID for approval
result = orchestrator2.approve_transfer("TRF-INVALID-999")
print(f"  Invalid approval: {result.get('error', 'N/A')}")

# Invalid transfer ID for explanation
result = orchestrator2.explain_transfer("TRF-INVALID-999")
print(f"  Invalid explanation: {result.get('error', result.get('error', 'Not found'))}")

print()
print("  All errors handled gracefully — no crashes, no exceptions.")


Error Handling Tests
  Invalid scenario: Invalid scenario: invalid_scenario
  Invalid SKU: No forecast found for SKU-NONEXIST at Dubai
  Invalid approval: Invalid transfer ID: TRF-INVALID-999
  Invalid explanation: Invalid transfer ID: TRF-INVALID-999

  All errors handled gracefully — no crashes, no exceptions.


## 12. Comprehensive System Summary

The `get_system_summary()` function provides a complete overview combining
all agent outputs.


In [25]:
# --- Get system summary ---
system_summary = orchestrator2.get_system_summary("demonstration")

print("Comprehensive System Summary — Demonstration Scenario")
print("=" * 70)
print(f"  Generated at: {system_summary['generated_at'][:19]}")
print()
print("  Forecast:")
fc = system_summary['forecast']
print(f"    Total 30d demand:     {fc['total_30d_demand']:.0f}")
print(f"    Combos:               {fc['num_combos']}")
print(f"    Avg uncertainty:      {fc['avg_uncertainty_pct']:.1%}")
print()
print("  Inventory:")
inv = system_summary['inventory']
print(f"    Risk counts:          {inv['risk_counts']}")
print(f"    Total unmet demand:   {inv['total_unmet_demand']:.0f}")
print(f"    Total excess stock:   {inv['total_excess_stock']:.0f}")
print(f"    Coverage ratio:       {inv['coverage_ratio']:.2f}x")
print()
print("  Optimization:")
opt = system_summary['optimization']
print(f"    Has recommendations:  {opt['has_recommendations']}")
print(f"    Num recommendations:  {opt['num_recommendations']}")
print(f"    Total units:          {opt['total_units_recommended']:,}")
print(f"    Total transfer cost:  {opt['total_transfer_cost']:,.2f} AED")
print(f"    Baseline unmet:       {opt['baseline_unmet']:.1f}")
print(f"    Post-transfer unmet:  {opt['post_transfer_unmet']:.1f}")
print(f"    Unmet avoided:        {opt['unmet_avoided']:.1f}")
print(f"    Reduction:            {opt['unmet_reduction_pct']:.1f}%")
print(f"    Net economic benefit: {opt['net_economic_benefit']:,.2f} AED")
print(f"    Shortages eliminated: {opt['shortages_fully_eliminated']}")
print()
print("  Approval:")
appr = system_summary['approval']
print(f"    Total:                {appr['total']}")
print(f"    Pending:              {appr['pending']}")
print(f"    Approved:             {appr['approved']}")
print(f"    Rejected:             {appr['rejected']}")
print()
print("  Explanation:")
print(f"    {system_summary['explanation'][:300]}...")


Comprehensive System Summary — Demonstration Scenario
  Generated at: 2026-10-27T15:14:48

  Forecast:
    Total 30d demand:     6385
    Combos:               60
    Avg uncertainty:      13.9%

  Inventory:
    Risk counts:          {'Critical': 32, 'Excess': 19, 'Healthy': 9}
    Total unmet demand:   1351
    Total excess stock:   5385
    Coverage ratio:       1.71x

  Optimization:
    Has recommendations:  True
    Num recommendations:  28
    Total units:          1,108
    Total transfer cost:  2,021.50 AED
    Baseline unmet:       1119.7
    Post-transfer unmet:  12.4
    Unmet avoided:        1107.3
    Reduction:            98.9%
    Net economic benefit: 44,817.17 AED
    Shortages eliminated: 2

  Approval:
    Total:                28
    Pending:              26
    Approved:             1
    Rejected:             1

  Explanation:
    The optimization agent identified 28 transfer recommendations for the demonstration scenario. These transfers are projected to reduce 

## 13. Agent Communication Log

All agent communications are logged in SQLite. The log can be exported to CSV
for auditing and debugging.


In [26]:
# --- Export agent log ---
log_path = os.path.join(DATA_DIR, "agent_workflow_log.csv")
orchestrator2.export_agent_log(log_path)

log_df = pd.read_csv(log_path)
print(f"Agent Workflow Log Exported: {log_path}")
print(f"Total log entries: {len(log_df)}")
print()
print("Log Summary by Agent:")
agent_counts = log_df['agent_name'].value_counts()
for agent, count in agent_counts.items():
    print(f"  {agent}: {count} messages")
print()
print("Log Summary by Task:")
task_counts = log_df['task'].value_counts()
for task, count in task_counts.items():
    print(f"  {task}: {count}")
print()
print("Sample log entries (first 10):")
print(log_df[['timestamp', 'agent_name', 'task', 'status']].head(10).to_string(index=False))


Agent Workflow Log Exported: C:\Users\antho\Desktop\CUD\Semester 7\Hackathon\Problem statement\Datasets\agent_workflow_log.csv
Total log entries: 47

Log Summary by Agent:
  ApprovalAgent: 17 messages
  InventoryRiskAgent: 8 messages
  ForecastingAgent: 7 messages
  OrchestratorAgent: 5 messages
  ExplanationAgent: 5 messages
  OptimizationAgent: 5 messages

Log Summary by Task:
  init: 12
  get_transfer_recommendations: 5
  analyze_inventory: 5
  get_approval_status: 4
  approve_transfer: 3
  explain_system_summary: 2
  get_approval_summary: 2
  summarize_forecast: 2
  get_demand_forecast: 2
  get_all_approvals: 2
  reject_transfer: 2
  get_system_summary: 1
  get_pending_approvals: 1
  explain_transfer: 1
  load_recommendations: 1
  compare_scenarios: 1
  explain_model_limitations: 1

Sample log entries (first 10):
                 timestamp         agent_name                         task  status
2026-10-27T15:14:48.383862  OrchestratorAgent           get_system_summary success
2026-

## 14. Run Validation Tests

Execute the full test suite to validate all agent functionality.


In [27]:
# --- Run the agent system tests ---
import subprocess

test_path = r"c:\Users\antho\CodeBuddy\20261027105634\agent_system_tests.py"
result = subprocess.run([sys.executable, test_path], capture_output=True, text=True, timeout=300)

print("Agent System Tests")
print("=" * 70)
print(result.stdout[-3000:] if len(result.stdout) > 3000 else result.stdout)
if result.stderr:
    print("STDERR:", result.stderr[:500])
print(f"\nExit code: {result.returncode}")


Agent System Tests
 Prevention ---
  [PASS] First approval succeeds: OK
  [PASS] Duplicate approval prevented: Transfer TRF-DEM-SKU-016-SHA-ABU is already approved.
  [PASS] Cannot reject approved transfer: Transfer TRF-DEM-SKU-016-SHA-ABU was already approved. Approval decisions are final.

--- Test: Invalid Transfer ID ---
  [PASS] Invalid ID approval rejected: Invalid transfer ID: TRF-INVALID-001
  [PASS] Invalid ID rejection rejected: Invalid transfer ID: TRF-INVALID-002
  [PASS] Invalid ID status not found: Invalid transfer ID: TRF-INVALID-003
  [PASS] Invalid ID explanation rejected: Invalid transfer ID: TRF-INVALID-004

--- Test: Error Handling ---
  [PASS] Invalid scenario returns error: Invalid scenario: invalid_scenario
  [PASS] Invalid scenario in optimization returns error: Invalid scenario: invalid_scenario
  [PASS] Invalid scenario in summary returns error: Invalid scenario: invalid_scenario
  [PASS] Invalid SKU forecast handled: No forecast found for SKU-NONEXIST at Duba

## Summary — Step 5: Agentic AI Orchestration

### Deliverables Created

| Deliverable | Description |
|---|---|
| `tradeflow_agents/` | Modular Python package (8 files) |
| `agent_system_tests.py` | 86 validation tests (all passing) |
| `Step5_Agentic_AI_Demo.ipynb` | This notebook (complete workflow demonstration) |
| `agent_workflow_log.csv` | Agent communication log exported from SQLite |

### Agent Architecture

| Agent | Role |
|---|---|
| **ForecastingAgent** | Loads XGBoost forecasts, retrieves per-SKU/warehouse predictions |
| **InventoryRiskAgent** | Analyzes shortages, excess stock, ageing; compares scenarios |
| **OptimizationAgent** | Runs scipy.optimize.milp, validates source inventory, respects lead times |
| **ExplanationAgent** | Generates deterministic explanations using actual optimization numbers |
| **ApprovalAgent** | Manages approval workflow with SQLite persistence, prevents duplicates |
| **OrchestratorAgent** | Coordinates all agents, supports individual and full-workflow calls |

### Key Features

- **Real operations** — no hardcoded responses; agents load data and run computations
- **SQLite persistence** — approval decisions survive restarts
- **Scenario isolation** — original and demonstration results kept separate
- **Lead-time aware** — transfers arriving after stockout are excluded
- **No double allocation** — source inventory capacity constraints enforced
- **Human approval** — all transfers start as Pending, never auto-approved
- **Deterministic explanations** — all figures from actual optimization output
- **Step 4 discrepancy documented** — 98.9% aggregate reduction vs 2/26 fully eliminated

### Important Restrictions Honored

- Existing forecasting models and datasets preserved
- No fabricated recommendations
- No automatic approvals
- No real-world transfer execution
- No hardcoded financial benefits
- All benefits labeled as projected estimates
- Streamlit dashboard not built (Step 6)

**Stop after Step 5. Report results.**
